In [1]:
!pip install ucimlrepo scikit-learn pandas numpy matplotlib seaborn joblib

In [2]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

# Fetch Phishing Websites dataset from UCI
phishing_data = fetch_ucirepo(id=327)

# Extract input features and target
X = phishing_data.data.features.copy()
y = phishing_data.data.targets.copy()

# Convert target into a Series
y = y.iloc[:, 0]

print("Dataset downloaded successfully!")

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature names:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

print("\nMissing values:", X.isnull().sum().sum())

Dataset downloaded successfully!

Feature shape: (11055, 30)
Target shape: (11055,)

Feature names:
['having_ip_address', 'url_length', 'shortining_service', 'having_at_symbol', 'double_slash_redirecting', 'prefix_suffix', 'having_sub_domain', 'sslfinal_state', 'domain_registration_length', 'favicon', 'port', 'https_token', 'request_url', 'url_of_anchor', 'links_in_tags', 'sfh', 'submitting_to_email', 'abnormal_url', 'redirect', 'on_mouseover', 'rightclick', 'popupwindow', 'iframe', 'age_of_domain', 'dnsrecord', 'web_traffic', 'page_rank', 'google_index', 'links_pointing_to_page', 'statistical_report']

Target distribution:
result
 1    6157
-1    4898
Name: count, dtype: int64

Missing values: 0


In [3]:
# Check dataset dimensions
assert X.shape[0] == 11055
assert X.shape[1] == 30
assert y.nunique() == 2

# Check duplicate records
print("Duplicate feature rows:", X.duplicated().sum())

# Check data types
print("\nData types:")
print(X.dtypes.value_counts())

# Check target classes
print("\nTarget classes:")
print(sorted(y.unique()))

# Display first five observations
display(X.head())

Duplicate feature rows: 5270

Data types:
int64    30
Name: count, dtype: int64

Target classes:
[np.int64(-1), np.int64(1)]


,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,rightclick,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,1,-1,-1,-1,-1,1,1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,1,-1,-1,0,-1,1,1,1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,1,-1,1,-1,1,0,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,1,-1,-1,1,-1,1,-1,1
4,1,0,-1,1,1,-1,1,1,-1,1,...,1,-1,1,-1,-1,0,-1,1,1,1


In [4]:
# Convert original labels into binary labels
# -1 → Phishing (1)
# 1 → Legitimate (0)


y = y.map({
    -1: 1,
     1: 0
})

# Rename target
y.name = "target"

print("Updated target distribution:")
print(y.value_counts())

print("\nTarget interpretation:")
print("1 = Phishing")
print("0 = Legitimate")

Updated target distribution:
target
0    6157
1    4898
Name: count, dtype: int64

Target interpretation:
1 = Phishing
0 = Legitimate


In [5]:
# Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training feature shape: (8844, 30)
Testing feature shape: (2211, 30)

Training target distribution:
target
0    4926
1    3918
Name: count, dtype: int64

Testing target distribution:
target
0    1231
1     980
Name: count, dtype: int64


In [6]:
# import libraries 

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

Define all five models

In [7]:
from sklearn.calibration import CalibratedClassifierCV

models = {

    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(
            random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]),

    "Naive Bayes": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", GaussianNB())
    ]),

    "Support Vector Machine": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", CalibratedClassifierCV(
            estimator=SVC(
                kernel="rbf",
                C=1.0,
                random_state=42
            ),
            method="sigmoid",
            cv=3
        ))
    ])

}

print("Five classification models defined successfully.")

Five classification models defined successfully.


In [8]:
# Training models
trained_models = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    trained_models[name] = model

    print(f"{name} trained successfully.")

print("\nAll five models trained successfully!")


Training Logistic Regression...
Logistic Regression trained successfully.

Training Decision Tree...
Decision Tree trained successfully.

Training KNN...
KNN trained successfully.

Training Naive Bayes...
Naive Bayes trained successfully.

Training Support Vector Machine...
Support Vector Machine trained successfully.

All five models trained successfully!


In [9]:

#Calculate Evaluation metrics for all 5 models

results = []

for name, model in trained_models.items():

    # Predict class labels
    y_pred = model.predict(X_test)

    # Predict probability of phishing class
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)

    auc = roc_auc_score(y_test, y_prob)

    precision = precision_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    mcc = matthews_corrcoef(y_test, y_pred)

    # Store results
    results.append({
        "ML Model": name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })


# Create comparison dataframe
results_df = pd.DataFrame(results)

# Display model comparison
print("MODEL PERFORMANCE COMPARISON")

display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "AUC": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}",
        "MCC": "{:.4f}"
    })
)

MODEL PERFORMANCE COMPARISON


,ML Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.9281,0.9785,0.9344,0.9010,0.9174,0.8542
1,Decision Tree,0.9697,0.9764,0.9760,0.9551,0.9654,0.9386
2,KNN,0.9471,0.9855,0.9499,0.9296,0.9397,0.8927
3,Naive Bayes,0.6160,0.9677,0.5359,0.9990,0.6975,0.4067
4,Support Vector Machine,0.9516,0.9877,0.9571,0.9327,0.9447,0.9019


In [10]:
#Save all five trained models

import os
import joblib

# Create model directory
os.makedirs("model", exist_ok=True)

# Define model filenames
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}

# Save all five trained models
for name, model in trained_models.items():

    file_path = os.path.join(
        "model",
        model_files[name]
    )

    joblib.dump(model, file_path)

    print(f"{name} saved successfully.")

print("\nAll five models saved successfully!")

Logistic Regression saved successfully.
Decision Tree saved successfully.
KNN saved successfully.
Naive Bayes saved successfully.
Support Vector Machine saved successfully.

All five models saved successfully!


Create test_data.csv

In [11]:
# Combine test features and target
test_data = X_test.copy()

test_data["target"] = y_test.values

# Save test dataset
test_data.to_csv("test_data.csv", index=False)

print("test_data.csv created successfully!")

print("\nTest dataset shape:", test_data.shape)

print("\nFirst five records:")
display(test_data.head())

test_data.csv created successfully!

Test dataset shape: (2211, 31)

First five records:


,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report,target
3588,1,-1,1,-1,1,-1,0,0,1,-1,...,-1,1,1,-1,-1,-1,1,0,-1,1
1688,1,1,1,1,1,-1,0,0,-1,1,...,1,1,1,-1,1,-1,-1,1,1,1
7280,-1,-1,1,1,1,-1,-1,-1,-1,1,...,1,1,-1,1,-1,1,1,1,1,1
2551,1,1,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,-1,-1,-1,0,1,1
235,1,-1,1,1,1,-1,-1,1,1,1,...,1,1,-1,1,-1,-1,1,0,1,0


Create requirements.txt

In [12]:
requirements = """streamlit
scikit-learn
numpy
pandas
matplotlib
seaborn
joblib
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

print("\nFile contents:")
print(requirements)

requirements.txt created successfully!

File contents:
streamlit
scikit-learn
numpy
pandas
matplotlib
seaborn
joblib



In [13]:
import sklearn
import streamlit
import numpy
import pandas
import joblib

print("scikit-learn:", sklearn.__version__)
print("streamlit:", streamlit.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("joblib:", joblib.__version__)

scikit-learn: 1.9.0
streamlit: 1.58.0
numpy: 2.4.6
pandas: 3.0.3
joblib: 1.5.3


In [14]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

st.set_page_config(
    page_title="Phishing Website Classification",
    page_icon="🔍",
    layout="wide"
)

st.title("Phishing Website Classification")
st.write("Classification of websites using five Machine Learning models.")

model_files = {
    "Logistic Regression": "model/logistic_regression.joblib",
    "Decision Tree": "model/decision_tree.joblib",
    "KNN": "model/knn.joblib",
    "Naive Bayes": "model/naive_bayes.joblib",
    "Support Vector Machine": "model/svm.joblib"
}

st.header("1. Upload Test Dataset")

uploaded_file = st.file_uploader(
    "Upload test_data.csv",
    type=["csv"]
)

if uploaded_file is not None:

    test_data = pd.read_csv(uploaded_file)

    st.success("Test dataset uploaded successfully.")

    st.write("Dataset shape:", test_data.shape)

    st.dataframe(test_data.head())

    if "target" not in test_data.columns:
        st.error("The uploaded CSV must contain a 'target' column.")
        st.stop()

    X_test = test_data.drop(columns=["target"])
    y_test = test_data["target"]

    st.header("2. Select Classification Model")

    selected_model = st.selectbox(
        "Choose a model:",
        list(model_files.keys())
    )

    model = joblib.load(model_files[selected_model])

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    st.header("3. Evaluation Metrics")

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric("Accuracy", f"{accuracy:.4f}")
        st.metric("Precision", f"{precision:.4f}")

    with col2:
        st.metric("AUC", f"{auc:.4f}")
        st.metric("Recall", f"{recall:.4f}")

    with col3:
        st.metric("F1 Score", f"{f1:.4f}")
        st.metric("MCC", f"{mcc:.4f}")

    st.header("4. Confusion Matrix")

    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    )

    st.dataframe(
        cm_df,
        use_container_width=True
    )

    st.write(f"Selected Model: **{selected_model}**")

    # --------------------------------------------------
    # Model Comparison
    # --------------------------------------------------

    st.header("5. Model Comparison")

    comparison_results = []

    for model_name, model_path in model_files.items():

        comparison_model = joblib.load(model_path)

        comparison_y_pred = comparison_model.predict(X_test)
        comparison_y_prob = comparison_model.predict_proba(X_test)[:, 1]

        comparison_results.append({
            "ML Model Name": model_name,
            "Accuracy": accuracy_score(
                y_test, comparison_y_pred
            ),
            "AUC": roc_auc_score(
                y_test, comparison_y_prob
            ),
            "Precision": precision_score(
                y_test, comparison_y_pred
            ),
            "Recall": recall_score(
                y_test, comparison_y_pred
            ),
            "F1": f1_score(
                y_test, comparison_y_pred
            ),
            "MCC": matthews_corrcoef(
                y_test, comparison_y_pred
            )
        })

    comparison_df = pd.DataFrame(comparison_results)

    st.dataframe(
        comparison_df.style.format({
            "Accuracy": "{:.4f}",
            "AUC": "{:.4f}",
            "Precision": "{:.4f}",
            "Recall": "{:.4f}",
            "F1": "{:.4f}",
            "MCC": "{:.4f}"
        }),
        use_container_width=True
    )

    # --------------------------------------------------
    # Overall Winner
    # --------------------------------------------------

    st.header("6. Overall Winner")

    # Calculate the average of the six evaluation metrics
    metric_columns = [
        "Accuracy",
        "AUC",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]

    comparison_df["Overall Score"] = (
        comparison_df[metric_columns].mean(axis=1)
    )

    winner_row = comparison_df.loc[
        comparison_df["Overall Score"].idxmax()
    ]

    winner_model = winner_row["ML Model Name"]
    winner_score = winner_row["Overall Score"]

    st.success(
        f"🏆 Overall Winner: **{winner_model}**"
    )

    st.write(
        f"Overall Score: **{winner_score:.4f}**"
    )

    st.write(
        "The overall winner is selected based on the highest "
        "average score across Accuracy, AUC, Precision, Recall, "
        "F1 Score and MCC."
    )

Overwriting app.py


In [15]:
#check that all required files exist

import os

print("app.py:", os.path.exists("app.py"))
print("test_data.csv:", os.path.exists("test_data.csv"))
print("requirements.txt:", os.path.exists("requirements.txt"))

print("\nModel files:")

model_files = [
    "model/logistic_regression.joblib",
    "model/decision_tree.joblib",
    "model/knn.joblib",
    "model/naive_bayes.joblib",
    "model/svm.joblib"
]

for file in model_files:
    print(file, "->", os.path.exists(file))

app.py: True
test_data.csv: True
requirements.txt: True

Model files:
model/logistic_regression.joblib -> True
model/decision_tree.joblib -> True
model/knn.joblib -> True
model/naive_bayes.joblib -> True
model/svm.joblib -> True


Stream lit

In [16]:
import subprocess
import sys

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.headless=true",
        "--server.port=8501"
    ]
)

print("Streamlit started successfully!")
print("Open: http://localhost:8501")

Streamlit started successfully!
Open: http://localhost:8501
